# Chapter 16 — Memory Is a Context Source

## Question

**If information has been remembered correctly, does that mean it belongs in the current context?**

Falsifiable structure: with a valid versioned store and a pinned present task, does the admission policy ever return the empty set as the correct structural outcome? If yes, durability never grants residency. This is a boundary notebook: no memory system is built, no extraction or consolidation is tested.

## Setup — durable store, selected context

`Memory is durable. Context is selected.` The Context book begins at memory candidates; everything before candidacy is consumed as interface, everything after admission is behaviour.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class SourceKind(Enum):
    MEMORY = 'MEMORY'
    RETRIEVAL = 'RETRIEVAL'
    FILE = 'FILE'
    TOOL = 'TOOL'
    AGENT_STATE = 'AGENT_STATE'

@dataclass(frozen=True)
class ContextCandidate:
    identity: str
    source_kind: SourceKind
    representation: str
    token_cost: int
    provenance: str
    scope: str
    validity: str

# Frozen valid versioned store. No extraction, consolidation, or trust mechanism varies below.
STORE = [
    ContextCandidate('dec-backend', SourceKind.MEMORY, 'PostgreSQL selected for event store', 120, 'adr-009', 'project', 'current'),
    ContextCandidate('fail-approach', SourceKind.MEMORY, 'SQLite attempt failed on serialisable writes', 100, 'adr-009', 'project', 'historical'),
    ContextCandidate('old-pref', SourceKind.MEMORY, 'old deploy preference: fridays', 60, 'note-114', 'project', 'stale'),
    ContextCandidate('redundant-fact', SourceKind.MEMORY, 'repo uses trunk-based development', 50, 'note-002', 'project', 'current'),
    ContextCandidate('background', SourceKind.MEMORY, '2024 migration retrospective', 400, 'note-087', 'project', 'historical'),
    ContextCandidate('hist-sqlite', SourceKind.MEMORY, 'SQLite was used before July', 60, 'adr-004', 'project', 'before-July'),
]
print(f'store holds {len(STORE)} valid items; none is in context yet.')

## Baseline — the durability table, generated from code

In [ ]:
states = [
    ('old decision in store', True, False, 'yes', 'no'),
    ('active plan in window', False, True, 'no', 'yes'),
    ('tool result in bundle', False, True, 'no', 'yes'),
    ('memory candidate at gate', True, False, 'yes', 'not yet'),
    ('admitted memory in bundle', True, True, 'by origin', 'yes'),
]
print(f"{'state':28s} {'durable':>7s} {'resident':>8s} {'memory':>9s} {'context':>7s}")
for name, dur, res, mem, ctx in states:
    print(f'{name:28s} {str(dur):>7s} {str(res):>8s} {mem:>9s} {ctx:>7s}')
assert states[0][1] and not states[0][2] and states[0][4] == 'no'
assert states[4][3] == 'by origin' and states[4][4] == 'yes'

## Intervention 1 — memory does not self-admit

In [ ]:
pool = list(STORE)  # memory adapter: same common candidate type, no subtype
context_ids = set()  # nothing admitted yet
assert all(isinstance(c, ContextCandidate) and c.source_kind is SourceKind.MEMORY for c in pool)
assert all(c.identity not in context_ids for c in pool)
print(f'store: {len(STORE)} items; pool: {len(pool)} candidates; context: {len(context_ids)} items')
print('Memory says X, therefore candidate X, therefore is X appropriate now — never therefore show X.')

## Intervention 2 — the present task pins first

Current instruction, fresh observation, and exact task constraint are pinned before memory competes. A highly relevant memory still does not displace a pinned instruction; authority conflicts in general belong to Chapter 19.

In [ ]:
BUDGET = 1000
pinned = {'instruction: do not modify the existing migration': 40,
          'fresh observation: migration test red': 60,
          'exact constraint: tenant 042 manual cutover': 50}
pinned_cost = sum(pinned.values())
remaining = BUDGET - pinned_cost
print(f'pinned non-negotiables: {pinned_cost} tokens; memory competes for the remaining {remaining}')
# Highly relevant historical memory admitted only within the remainder, never by displacing pins.
admitted = ['dec-backend', 'fail-approach']
admitted_cost = sum(next(c.token_cost for c in STORE if c.identity == i) for i in admitted)
assert admitted_cost <= remaining
assert set(pinned) == {'instruction: do not modify the existing migration',
                         'fresh observation: migration test red',
                         'exact constraint: tenant 042 manual cutover'}
print(f'admitted memory: {admitted} ({admitted_cost} tokens); pins untouched.')

## Intervention 3 — null admission is a success path

In [ ]:
task = 'rewrite this supplied string: hello world'
admitted_none = []
print(f'task: {task!r}; memory admitted: {len(admitted_none)}')
assert admitted_none == []
print('Restraint is a capability, not a retrieval failure.')

## Observation — the chain with honest ends

In [ ]:
chain = {'remembered': 'dec-backend in store', 'candidate': 'dec-backend in pool',
         'admitted': 'dec-backend in bundle', 'used': 'NOT_MEASURED', 'helpful': 'NOT_MEASURED'}
for stage, value in chain.items():
    print(f'{stage:10s} {value}')
assert chain['used'] == 'NOT_MEASURED' and chain['helpful'] == 'NOT_MEASURED'

# Validity metadata travels so later machinery can distinguish; nothing is resolved here.
print('hist-sqlite validity=before-July; dec-backend validity=current (Chapter 20 owns freshness).')
assert next(c.validity for c in STORE if c.identity == 'hist-sqlite') == 'before-July'

## Sibling-book results (imported, not reproduced)

> SUPPORTED SIBLING-BOOK RESULT (Memory book, Ch12, frozen run `ch12-20260920T204414Z-behavior`; nine controlled plus five transfer tasks, synthetic fixtures, small readers): assembled structured memory raised success 0.226 → 0.488 vs 0.393 for strong retrieval alone; decisive-memory removal dropped the intervention subset to 0.250 and restoration lifted it to 0.778; deliberately wrong memory averaged 0.042 with harmful actions in two of four tasks; a memory-independent echo task was harmed by supplied memory. Scope: those fixtures, readers, and tasks only — no general law is imported.

The executable notebook above tests none of those numbers. It tests the boundary they cross: candidacy into admission.

## Try it

1. Shrink BUDGET below the pinned total and confirm the policy breaks loudly instead of silently dropping a pin.
2. Admit `old-pref` (stale validity) and note the notebook cannot stop you: staleness resolution is Chapter 20's job.
3. Run the null-admission cell against a memory-diagnostic task and confirm the chain reports admitted-but-absent rather than success.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print([c.identity for c in STORE if c.validity == 'current'])

## What this demonstrates

- Storage and durability do not grant current-context residency: six valid items, zero admitted until a gate decides.
- Memory uses the same candidate/admission interface as every other source: one `ContextCandidate` type, `source_kind` as metadata.
- Admitting no memory can be the correct context decision.

## What this does not demonstrate

- That memory improves the current task, or deserves a dedicated admission policy.
- That memory has inherent authority, or that relevance makes the remembered current.
- That extraction, consolidation, or truth maintenance were tested here.
- That sibling-book numbers generalise beyond their stated fixtures and readers.

## Connection to the chapter

Memory is one context producer, and it asks what the past may still decide:

> Memory is one context producer. Tools are another — but tools impose context cost both before and after they execute.

That is Chapter 17.